# Loan Default / Credit Risk Prediction — Real Data Edition

**Business problem:** Predict whether a loan applicant is likely to default so a lender can assess credit risk more consistently.

**What changed from the previous version:** The earlier notebook used a **synthetic (randomly generated)** dataset. This version replaces it with **two real, publicly available loan datasets from Kaggle**, combined into one larger real-world dataset, following this pipeline:

```
Filtering  ->  Dataset Create  ->  Realistic check  ->  Clean & Preprocess
   ->  Feature Engineering  ->  Feature Selection  ->  Train/Test Split (70/30)
   ->  Train 3 Models  ->  Evaluate  ->  Feature Importance  ->  Predict new applicant
```

**Data sources used:**
1. `credit_risk_dataset.csv` — 32,581 borrowers, 12 columns (age, income, home ownership, loan grade, loan amount, interest rate, default status, etc.)
2. `Loan_Default_csv.xlsx` — 148,670 mortgage/loan applicants, 34 columns (much messier — many missing values, categorical age bands, etc.)

**Approach:** Build and compare three binary-classification models:
1. **Logistic Regression** — interpretable statistical baseline
2. **Decision Tree** — rule-based model, easy to visualize
3. **Random Forest** — ensemble of decision trees for more complex patterns

Every calculation step below prints intermediate numbers (row counts, medians used for imputation, correlation values, etc.) so the full logic is visible, not hidden inside a black box.

## 1. Install & import libraries

Run this cell first in Google Colab. The installation command is harmless if the packages are already installed.

In [ ]:
!pip install -q numpy pandas matplotlib seaborn scikit-learn ipywidgets openpyxl

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

pd.set_option('display.width', 140)
np.random.seed(42)
sns.set_style("whitegrid")

## 2. Load the two real datasets

Upload `credit_risk_dataset.csv` and `Loan_Default_csv.xlsx` to your Colab session (or Google Drive), then load them.

In [ ]:
df1_raw = pd.read_csv("credit_risk_dataset.csv")
df2_raw = pd.read_excel("Loan_Default_csv.xlsx")

print("Dataset 1 (credit_risk_dataset.csv):", df1_raw.shape)
print("Dataset 2 (Loan_Default_csv.xlsx):  ", df2_raw.shape)

print("\nDataset 1 columns:", list(df1_raw.columns))
print("\nDataset 2 columns:", list(df2_raw.columns))

print("\nDataset 1 missing values:")
print(df1_raw.isna().sum())
print("\nDataset 2 missing values (top 10 by count):")
print(df2_raw.isna().sum().sort_values(ascending=False).head(10))

## 3. Filtering

Two things happen here:
- **Row filtering** — remove rows that are physically impossible or unusable (e.g. negative income, age outside a realistic working-age range, employment length longer than a person's working life).
- **Column filtering** — Dataset 2 has 34 columns, many of them mostly empty (e.g. `Upfront_charges` is missing for ~27% of rows) or not relevant to credit risk (e.g. `Security_Type`, `construction_type`). We keep only the columns that are usable and conceptually map onto Dataset 1's features.

In [ ]:
# --- Row filtering: Dataset 1 ---
df1 = df1_raw.copy()
before = len(df1)
df1 = df1[(df1['person_age'] >= 18) & (df1['person_age'] <= 90)]
df1 = df1[df1['person_income'] > 0]
df1 = df1[df1['person_emp_length'].isna() | (df1['person_emp_length'] <= (df1['person_age'] - 14))]
df1 = df1.reset_index(drop=True)
print(f"Dataset 1: removed {before - len(df1)} invalid rows ({before} -> {len(df1)})")

# --- Column filtering: Dataset 2 ---
# Keep only columns that map onto credit-risk concepts and aren't mostly empty
keep_cols_df2 = ['age', 'income', 'loan_amount', 'rate_of_interest', 'Credit_Score', 'dtir1', 'Status']
df2 = df2_raw[keep_cols_df2].copy()

# --- Row filtering: Dataset 2 ---
before = len(df2)
df2 = df2[df2['income'] > 0]
df2 = df2.reset_index(drop=True)
print(f"Dataset 2: removed {before - len(df2)} invalid rows ({before} -> {len(df2)})")
print(f"Dataset 2: kept {len(keep_cols_df2)} of {df2_raw.shape[1]} original columns")

## 4. Dataset Create — harmonize schema & combine into one real dataset

The two datasets don't use the same column names, units, or encodings, so each is mapped onto a **shared schema** before combining:

| Shared feature | Dataset 1 source | Dataset 2 source | Notes |
|---|---|---|---|
| `age` | `person_age` (already numeric) | `age` (text ranges like `"25-34"`) | Dataset 2's age bands are converted to their numeric midpoint |
| `annual_income` | `person_income` (already annual) | `income` (monthly) | Dataset 2's income is annualized: `income x 12` |
| `loan_amount` | `loan_amnt` | `loan_amount` | Same concept, different name |
| `interest_rate` | `loan_int_rate` | `rate_of_interest` | Same concept, different name |
| `employment_length_years` | `person_emp_length` | *not available* | Left missing for Dataset 2, handled in cleaning |
| `credit_score_raw` | `loan_grade` (A-G letters) | `Credit_Score` (500-900 numeric) | Dataset 1's letter grade is mapped to an approximate numeric score |
| `debt_to_income_pct` | `loan_percent_income x 100` | `dtir1` | Approximate equivalents — both express loan burden relative to income |
| `target` | `loan_status` | `Status` | Both already coded `1 = default, 0 = no default` |

**Important, honest caveat:** the `credit_score_raw` and `debt_to_income_pct` mappings across the two sources are *approximations*, not identical measurements. This is normal and expected when combining real datasets from different institutions — it's worth stating explicitly in a real project rather than hiding it.

In [ ]:
def grade_to_score(g):
    """Approximate mapping from letter loan grade (A-G) to a numeric credit-score-like value."""
    mapping = {'A': 750, 'B': 700, 'C': 650, 'D': 600, 'E': 550, 'F': 500, 'G': 450}
    return mapping.get(g, np.nan)

def age_range_to_midpoint(a):
    """Convert Dataset 2's text age bands into a single numeric midpoint."""
    mapping = {'<25': 22, '25-34': 29.5, '35-44': 39.5, '45-54': 49.5,
               '55-64': 59.5, '65-74': 69.5, '>74': 80}
    return mapping.get(a, np.nan)

# Harmonize Dataset 1
h1 = pd.DataFrame()
h1['source'] = ['dataset_1_credit_risk'] * len(df1)
h1['age'] = df1['person_age'].astype(float)
h1['annual_income'] = df1['person_income'].astype(float)
h1['loan_amount'] = df1['loan_amnt'].astype(float)
h1['interest_rate'] = df1['loan_int_rate']
h1['employment_length_years'] = df1['person_emp_length']
h1['credit_score_raw'] = df1['loan_grade'].apply(grade_to_score)
h1['debt_to_income_pct'] = df1['loan_percent_income'] * 100
h1['target'] = df1['loan_status']

# Harmonize Dataset 2
h2 = pd.DataFrame()
h2['source'] = ['dataset_2_loan_default'] * len(df2)
h2['age'] = df2['age'].apply(age_range_to_midpoint)
h2['annual_income'] = df2['income'] * 12          # monthly -> annual
h2['loan_amount'] = df2['loan_amount'].astype(float)
h2['interest_rate'] = df2['rate_of_interest']
h2['employment_length_years'] = np.nan             # not available in Dataset 2
h2['credit_score_raw'] = df2['Credit_Score'].astype(float)
h2['debt_to_income_pct'] = df2['dtir1']
h2['target'] = df2['Status']

# Combine (stack) both real datasets into one
combined = pd.concat([h1, h2], ignore_index=True)

print("Combined real dataset shape:", combined.shape)
print("\nRows contributed by each source:")
print(combined['source'].value_counts())
print("\nMissing values before cleaning:")
print(combined.isna().sum())

## 5. Realistic check

A quick sanity check that the combined dataset behaves like real financial data: reasonable ranges, a sensible default rate, and both sources represented.

In [ ]:
print("Overall default rate:", round(combined['target'].mean() * 100, 2), "%")
print("Default rate by source:")
print(combined.groupby('source')['target'].mean().round(3))
print("\nAge range:", combined['age'].min(), "-", combined['age'].max())
print("Annual income range:", combined['annual_income'].min(), "-", combined['annual_income'].max())
print("Loan amount range:", combined['loan_amount'].min(), "-", combined['loan_amount'].max())

## 6. Clean & Preprocess

Steps, each printed explicitly:
1. Remove any remaining rows with impossible values.
2. Add a **missing-value flag** for `employment_length_years` (mostly missing because Dataset 2 doesn't report it) — this preserves the information that it was missing, which can itself be predictive.
3. **Impute** missing numeric values with the column median (median is used instead of mean because these financial variables are skewed by outliers).
4. **Clip outliers** at the 1st/99th percentile so a few extreme values don't distort the models.

In [ ]:
# 1. Remove remaining invalid rows
before = len(combined)
combined = combined[combined['annual_income'] > 0]
combined = combined[(combined['age'] >= 18) & (combined['age'] <= 90)]
combined = combined[combined['loan_amount'] > 0]
combined = combined.reset_index(drop=True)
print(f"Rows after final validity check: {before} -> {len(combined)} ({len(combined)/before*100:.1f}% retained)")

# 2. Missing-value flag
combined['employment_length_known'] = combined['employment_length_years'].notna().astype(int)

# 3. Median imputation
medians = {}
for col in ['interest_rate', 'employment_length_years', 'debt_to_income_pct', 'credit_score_raw']:
    med = combined[col].median()
    medians[col] = med
    combined[col] = combined[col].fillna(med)

print("\nMedians used for imputation:")
for k, v in medians.items():
    print(f"  {k}: {v:.2f}")

# 4. Outlier clipping (1st-99th percentile)
print("\nOutlier clipping:")
for col in ['annual_income', 'loan_amount', 'interest_rate', 'debt_to_income_pct']:
    q1, q99 = combined[col].quantile([0.01, 0.99])
    n_clipped = ((combined[col] < q1) | (combined[col] > q99)).sum()
    combined[col] = combined[col].clip(q1, q99)
    print(f"  {col}: clipped {n_clipped} outlier rows to range [{q1:.1f}, {q99:.1f}]")

print("\nMissing values after cleaning (should all be 0):")
print(combined.isna().sum())
print("\nFinal cleaned shape:", combined.shape)
combined.describe().round(2)

## 7. Quick exploratory analysis on the real, combined dataset

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.boxplot(x="target", y="credit_score_raw", data=combined, ax=axes[0])
axes[0].set_title("Credit Score vs Default")
axes[0].set_xlabel("Default (0 = No, 1 = Yes)")

sns.boxplot(x="target", y="debt_to_income_pct", data=combined, ax=axes[1])
axes[1].set_title("Debt-to-Income % vs Default")
axes[1].set_xlabel("Default (0 = No, 1 = Yes)")

plt.tight_layout()
plt.show()

## 8. Feature Engineering

Creating new features that are often more predictive than raw columns on their own:
- **`loan_to_income_ratio`** = loan amount / annual income — a classic affordability signal
- **`credit_score_normalized`** = credit score rescaled to a common 0-100 range, since the two sources used different original scales
- **`high_dti_flag`** = 1 if debt-to-income is above 40% (a common lending risk threshold), else 0
- **`age_group`** = age bucketed into ranges, useful for later analysis/visualization

In [ ]:
combined['loan_to_income_ratio'] = combined['loan_amount'] / combined['annual_income']

cs_min, cs_max = combined['credit_score_raw'].min(), combined['credit_score_raw'].max()
combined['credit_score_normalized'] = (combined['credit_score_raw'] - cs_min) / (cs_max - cs_min) * 100
print(f"credit_score_raw range [{cs_min:.0f}, {cs_max:.0f}] rescaled to credit_score_normalized [0, 100]")

combined['high_dti_flag'] = (combined['debt_to_income_pct'] > 40).astype(int)

combined['age_group'] = pd.cut(
    combined['age'],
    bins=[17, 25, 35, 45, 55, 65, 91],
    labels=['18-25', '26-35', '36-45', '46-55', '56-65', '66+']
)

print("\nEngineered feature summary:")
print(combined[['loan_to_income_ratio', 'credit_score_normalized', 'high_dti_flag']].describe().round(3))
print("\nApplicants per age group:")
print(combined['age_group'].value_counts().sort_index())

## 9. Feature Selection

Two checks, both printed:
1. **Correlation with the target** — a simple, transparent first look at which features move with default risk.
2. **Random Forest feature importance** — a more robust, non-linear measure of how useful each feature is for prediction.

The top features by importance are kept for modeling. `source` is intentionally excluded from modeling — including it would let a model "cheat" by learning which dataset a row came from rather than learning genuine default-risk patterns.

In [ ]:
candidate_features = [
    'age', 'annual_income', 'loan_amount', 'interest_rate',
    'employment_length_years', 'employment_length_known',
    'credit_score_normalized', 'debt_to_income_pct', 'high_dti_flag',
    'loan_to_income_ratio'
]

print("Correlation of each candidate feature with the target (default):")
corr_with_target = (
    combined[candidate_features + ['target']]
    .corr()['target'].drop('target')
    .sort_values(key=abs, ascending=False)
)
print(corr_with_target.round(3))

X_temp = combined[candidate_features]
y_temp = combined['target']

rf_selector = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_selector.fit(X_temp, y_temp)

importances = pd.Series(rf_selector.feature_importances_, index=candidate_features).sort_values(ascending=False)
print("\nRandom Forest feature importance (used for selection):")
print(importances.round(4))

TOP_N = 8
selected_features = importances.head(TOP_N).index.tolist()
print(f"\nSelected top {TOP_N} features for modeling:")
print(selected_features)

## 10. Train / test split (70 / 30)

The same 70/30 split is used for all three models so their results are directly comparable. Logistic Regression uses standardization because it is sensitive to feature scale; the tree-based models do not require it.

In [ ]:
feature_cols = selected_features

X = combined[feature_cols]
y = combined['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train), f"({len(X_train)/len(X)*100:.0f}%)")
print("Testing rows: ", len(X_test), f"({len(X_test)/len(X)*100:.0f}%)")
print("Features used:", feature_cols)

## 11. Train 3 machine learning models

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=25,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    )
}

predictions = {}
probabilities = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions[name] = model.predict(X_test)
    probabilities[name] = model.predict_proba(X_test)[:, 1]

print("All 3 models trained successfully on the real, combined dataset.")

## 12. Compare model performance

- **Accuracy** — overall proportion of correct predictions
- **Precision** — among predicted defaults, how many actually defaulted
- **Recall** — among actual defaults, how many were identified
- **F1-score** — balance between precision and recall
- **ROC-AUC** — ability to distinguish defaulters from non-defaulters across thresholds

In [ ]:
results = []

for name in models:
    y_pred = predictions[name]
    y_proba = probabilities[name]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba)
    })

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False)
display(results_df.round(3))

## 13. Classification reports and confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, name in zip(axes, models):
    cm = confusion_matrix(y_test, predictions[name])
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
        xticklabels=["No Default", "Default"],
        yticklabels=["No Default", "Default"]
    )
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()

for name in models:
    print("=" * 70)
    print(name)
    print(classification_report(
        y_test, predictions[name],
        target_names=["No Default", "Default"],
        zero_division=0
    ))

## 14. ROC curves for all 3 models

In [ ]:
plt.figure(figsize=(7, 5))

for name in models:
    fpr, tpr, _ = roc_curve(y_test, probabilities[name])
    auc = roc_auc_score(y_test, probabilities[name])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="Random baseline")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.tight_layout()
plt.show()

## 15. Cross-validation check

A single train/test split can be lucky or unlucky. **5-fold cross-validation** retrains each model 5 times on different slices of the data and reports the average ROC-AUC, giving a more reliable estimate of how each model performs than one split alone.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
    cv_results.append({
        "Model": name,
        "Mean CV ROC-AUC": scores.mean(),
        "Std Dev": scores.std(),
        "Fold Scores": [round(s, 3) for s in scores]
    })
    print(f"{name}: mean ROC-AUC = {scores.mean():.3f} (+/- {scores.std():.3f}) across 5 folds")

cv_results_df = pd.DataFrame(cv_results)[["Model", "Mean CV ROC-AUC", "Std Dev"]]
display(cv_results_df.round(3))
print("\nA small standard deviation across folds means the model's performance is stable")
print("and not just a result of how this particular train/test split happened to fall.")

## 16. Handling class imbalance

Only about 24% of applicants in the combined dataset actually defaulted. Left as-is, models can reach high accuracy just by predicting "no default" most of the time — which is exactly why Logistic Regression's **recall** was so low in the earlier results (it missed most real defaulters).

Here each model is retrained with **`class_weight="balanced"`**, which makes mistakes on the minority class (default) count more during training, pushing the model to actually try to catch defaulters rather than ignore them. Decision Tree and Random Forest are retrained the same way; Logistic Regression's coefficients change as well.

In [ ]:
balanced_models = {
    "Logistic Regression (balanced)": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"))
    ]),
    "Decision Tree (balanced)": DecisionTreeClassifier(
        max_depth=6, min_samples_leaf=25, random_state=42, class_weight="balanced"
    ),
    "Random Forest (balanced)": RandomForestClassifier(
        n_estimators=300, max_depth=10, min_samples_leaf=10,
        random_state=42, n_jobs=-1, class_weight="balanced"
    )
}

balanced_results = []
for name, model in balanced_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    balanced_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, proba)
    })

balanced_results_df = pd.DataFrame(balanced_results)
print("Original (unbalanced) results, for comparison:")
display(results_df.round(3))
print("\nBalanced (class_weight='balanced') results:")
display(balanced_results_df.round(3))
print("\nNotice recall typically rises with class weighting — the models catch more real")
print("defaulters — usually at some cost to precision. This is a business trade-off, not")
print("purely a technical one: missing a defaulter (false negative) usually costs a lender")
print("more than incorrectly flagging a safe applicant (false positive).")

## 17. Feature importance

Different models provide different forms of feature importance:
- **Logistic Regression:** standardized coefficients indicate the direction and relative strength of each relationship
- **Decision Tree:** impurity-based importance shows which variables the tree split on most
- **Random Forest:** average feature importance across all trees

In [ ]:
logistic_model = models["Logistic Regression"].named_steps["model"]

coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": logistic_model.coef_[0]
}).sort_values("Coefficient")

plt.figure(figsize=(8, 5))
plt.barh(coef_df["Feature"], coef_df["Coefficient"])
plt.axvline(0, linewidth=0.8)
plt.title("Logistic Regression Coefficients")
plt.xlabel("Standardized coefficient")
plt.tight_layout()
plt.show()

importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Decision Tree": models["Decision Tree"].feature_importances_,
    "Random Forest": models["Random Forest"].feature_importances_
}).sort_values("Random Forest", ascending=True)

importance_df.set_index("Feature").plot.barh(figsize=(8, 5))
plt.title("Feature Importance — Decision Tree vs Random Forest")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

display(importance_df.sort_values("Random Forest", ascending=False).round(4))

## 18. Raw-input helper function

So far, the "new applicant" test required typing already-*engineered* values (like `loan_to_income_ratio` or `credit_score_normalized`), which isn't how a real user would enter data. This function instead takes the **raw, human-readable inputs** — the same kind of fields a loan officer would actually type in — and automatically computes the engineered features behind the scenes (the same formulas from Section 8), so the input side matches real-world data entry.

**Raw input columns:**
- `age` (years)
- `annual_income` (in currency units)
- `loan_amount` (in currency units)
- `interest_rate` (%)
- `credit_score` (roughly 500-900 scale, matching Dataset 2's scoring; a letter-grade applicant can be approximated using the same A-G mapping from Section 4)
- `debt_to_income_pct` (%)
- `employment_length_years` (optional — leave as `None` if unknown)

In [ ]:
def raw_to_model_features(age, annual_income, loan_amount, interest_rate,
                           credit_score, debt_to_income_pct,
                           employment_length_years=None):
    """Convert raw, human-entered applicant data into the engineered
    feature set the models were trained on."""

    loan_to_income_ratio = loan_amount / annual_income
    credit_score_normalized = (credit_score - cs_min) / (cs_max - cs_min) * 100
    credit_score_normalized = np.clip(credit_score_normalized, 0, 100)
    high_dti_flag = int(debt_to_income_pct > 40)

    if employment_length_years is None:
        employment_length_known = 0
        employment_length_years = medians['employment_length_years']
    else:
        employment_length_known = 1

    row = {
        "age": age,
        "annual_income": annual_income,
        "loan_amount": loan_amount,
        "interest_rate": interest_rate,
        "employment_length_years": employment_length_years,
        "employment_length_known": employment_length_known,
        "credit_score_raw": credit_score,
        "credit_score_normalized": credit_score_normalized,
        "debt_to_income_pct": debt_to_income_pct,
        "high_dti_flag": high_dti_flag,
        "loan_to_income_ratio": loan_to_income_ratio,
    }
    return pd.DataFrame([row])[feature_cols]

## 19. Test the models on a new loan applicant (raw input columns)

In [ ]:
new_applicant_raw = raw_to_model_features(
    age=34,
    annual_income=640000,
    loan_amount=350000,
    interest_rate=11.5,
    credit_score=560,
    debt_to_income_pct=45,
    employment_length_years=3
)

print("Engineered features computed from the raw input:")
display(new_applicant_raw)

new_results = []
for name, model in models.items():
    pred = model.predict(new_applicant_raw)[0]
    proba = model.predict_proba(new_applicant_raw)[0, 1]
    new_results.append({
        "Model": name,
        "Prediction": "DEFAULT RISK" if pred == 1 else "LIKELY TO REPAY",
        "Estimated Probability of Default": round(proba * 100, 1)
    })

display(pd.DataFrame(new_results))

## 20. Interactive demo — enter raw applicant data

Move the sliders to describe a hypothetical applicant using **raw, real-world fields**. Engineered features are computed automatically, and predictions from all three models update together.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

age_w = widgets.IntSlider(value=34, min=18, max=90, step=1, description="Age")
income_w = widgets.IntSlider(value=640000, min=15000, max=1000000, step=5000, description="Annual income")
loan_w = widgets.IntSlider(value=350000, min=2000, max=900000, step=5000, description="Loan amount")
ir_w = widgets.FloatSlider(value=11.5, min=3, max=17, step=0.1, description="Interest rate (%)")
credit_w = widgets.IntSlider(value=560, min=450, max=900, step=5, description="Credit score")
dti_w = widgets.FloatSlider(value=45, min=0, max=70, step=1, description="DTI (%)")
emp_w = widgets.FloatSlider(value=3, min=0, max=40, step=1, description="Employment yrs")
emp_known_w = widgets.ToggleButtons(value=True, options=[("Known", True), ("Unknown", False)], description="Emp. yrs known?")

out = widgets.Output()

def update_demo(*args):
    with out:
        clear_output(wait=True)
        emp_years = emp_w.value if emp_known_w.value else None

        applicant = raw_to_model_features(
            age=age_w.value,
            annual_income=income_w.value,
            loan_amount=loan_w.value,
            interest_rate=ir_w.value,
            credit_score=credit_w.value,
            debt_to_income_pct=dti_w.value,
            employment_length_years=emp_years
        )

        rows = []
        for name, model in models.items():
            pred = model.predict(applicant)[0]
            proba = model.predict_proba(applicant)[0, 1]
            rows.append({
                "Model": name,
                "Prediction": "DEFAULT RISK" if pred == 1 else "LIKELY TO REPAY",
                "Probability of Default (%)": round(proba * 100, 1)
            })
        print("Raw inputs entered:")
        print(f"  Age: {age_w.value} | Income: {income_w.value} | Loan: {loan_w.value} | "
              f"Rate: {ir_w.value}% | Credit score: {credit_w.value} | DTI: {dti_w.value}% | "
              f"Employment yrs: {'unknown' if emp_years is None else emp_years}")
        display(pd.DataFrame(rows))

for w in [age_w, income_w, loan_w, ir_w, credit_w, dti_w, emp_w, emp_known_w]:
    w.observe(update_demo, names="value")

ui = widgets.VBox([age_w, income_w, loan_w, ir_w, credit_w, dti_w, emp_w, emp_known_w])
display(ui, out)
update_demo()

## 21. Cost-benefit analysis (business impact)

Accuracy and F1-score don't mean much to a credit committee. What matters is **money**. Every wrong prediction has a different real-world cost:

- **False Negative** (model says "safe", applicant actually defaults) — the lender loses roughly the **entire loan amount**, since the borrower stops repaying.
- **False Positive** (model says "risky", applicant would actually have repaid) — the lender loses only the **profit that would have been earned**, i.e. the interest income on that loan, because a genuinely creditworthy applicant is rejected or priced out.

This is why False Negatives are usually far more expensive than False Positives in lending — which is also why the class-imbalance section above (Section 16) matters commercially, not just statistically.

In [ ]:
avg_loan_amount = combined['loan_amount'].mean()
avg_interest_rate = combined['interest_rate'].mean() / 100

cost_per_false_negative = avg_loan_amount                          # lender loses the loan principal
cost_per_false_positive = avg_loan_amount * avg_interest_rate       # lender loses the interest income

print(f"Average loan amount in the dataset: {avg_loan_amount:,.0f}")
print(f"Average interest rate: {avg_interest_rate*100:.2f}%")
print(f"Estimated cost per False Negative (missed defaulter): {cost_per_false_negative:,.0f}")
print(f"Estimated cost per False Positive (wrongly rejected applicant): {cost_per_false_positive:,.0f}")

cost_summary = []
all_model_sets = {**models, **balanced_models}

for name, model in all_model_sets.items():
    pred = model.predict(X_test)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    total_cost = fn * cost_per_false_negative + fp * cost_per_false_positive
    cost_summary.append({
        "Model": name,
        "False Negatives": fn,
        "False Positives": fp,
        "Estimated Cost": total_cost
    })

cost_df = pd.DataFrame(cost_summary).sort_values("Estimated Cost")
cost_df["Estimated Cost"] = cost_df["Estimated Cost"].round(0)
display(cost_df)
print("\nLower estimated cost is better. This reframes 'model accuracy' as 'money saved',")
print("which is the number a lending business actually cares about.")

## 22. Risk tiering

Real lenders rarely use a single "approve/reject" cutoff. Instead, applicants are grouped into **risk tiers** based on their predicted probability of default, and each tier gets different treatment — e.g. standard interest rate, a higher rate to offset risk, or manual review.

Tiers used here (a common simple structure):
- **Low risk:** probability of default < 20%
- **Medium risk:** 20% - 50%
- **High risk:** > 50%

In [ ]:
best_model_name = results_df.iloc[0]['Model']   # best model by ROC-AUC from Section 12
best_model = models[best_model_name]
test_probs = best_model.predict_proba(X_test)[:, 1]

def assign_tier(p):
    if p < 0.20:
        return "Low risk"
    elif p < 0.50:
        return "Medium risk"
    else:
        return "High risk"

tiers = pd.Series([assign_tier(p) for p in test_probs], name="Risk Tier")
tier_summary = pd.DataFrame({"Risk Tier": tiers, "Actual Default": y_test.values})

tier_table = tier_summary.groupby("Risk Tier").agg(
    Applicants=("Actual Default", "count"),
    Actual_Default_Rate=("Actual Default", "mean")
).reindex(["Low risk", "Medium risk", "High risk"])

tier_table["Actual_Default_Rate"] = (tier_table["Actual_Default_Rate"] * 100).round(1)
print(f"Risk tiers built using: {best_model_name} (best model by ROC-AUC)")
display(tier_table)

print("\nA well-behaved model should show the actual default rate clearly increasing")
print("from Low -> Medium -> High risk tiers, confirming the tiers are meaningful.")
print("\nExample business use: a lender could charge Low-risk tiers the standard rate,")
print("Medium-risk tiers a higher rate to offset expected losses, and send High-risk")
print("applicants to manual underwriting rather than an automatic decision.")

## 23. Executive summary (non-technical)

A short summary written for a credit committee or business stakeholder rather than a data scientist — the kind of paragraph that would open a report to non-technical decision-makers.

In [ ]:
best_row = results_df.iloc[0]

summary_text = f"""
EXECUTIVE SUMMARY

This project built a loan default prediction model using {len(combined):,} real loan
applications combined from two public lending datasets. Three modeling approaches were
tested; the best performer, {best_row['Model']}, correctly distinguishes likely defaulters
from likely repayers {best_row['ROC-AUC']*100:.1f}% of the time (ROC-AUC), with an accuracy
of {best_row['Accuracy']*100:.1f}% on applicants the model had never seen before.

The model was also tested for fairness of its cost trade-offs: missing a real defaulter
is treated as more costly to the business than mistakenly flagging a safe applicant,
matching how lenders actually experience risk. Applicants can further be grouped into
Low / Medium / High risk tiers to support different lending decisions (standard approval,
risk-based pricing, or manual review) rather than a single automatic yes/no.

Limitation: the two source datasets used slightly different definitions for credit score
and debt-to-income, which were approximated onto a common scale. Before real-world use,
this model would need a single consistent data pipeline, fairness testing across
demographic groups, and regulatory review.
"""

print(summary_text)

## 24. Regulatory & compliance note

A brief note on the real-world governance a model like this would need before being used for actual lending decisions — relevant given the data reflects an Indian lending context:

- **Fair lending:** the model must not directly or indirectly discriminate based on protected characteristics (e.g. gender, religion, caste). Sensitive attributes should be excluded from model features, and disparate-impact testing should be run across demographic groups before deployment — this notebook's feature set intentionally avoids protected characteristics, but disparate-impact testing was not performed here and would be a required next step.
- **RBI algorithmic lending guidance:** the Reserve Bank of India has emphasized transparency, explainability, and human oversight in digital lending and algorithm-driven credit decisions — meaning a model like this should support human review, not fully replace it, especially for High-risk or borderline cases.
- **Data privacy (DPDP Act, 2023):** since this project uses real personal financial data, any production use would need explicit consent for data collection, purpose limitation, secure storage, and a defined data retention/deletion policy.
- **Model governance:** in practice, a model like this would need periodic re-validation (since borrower behavior changes over time), a documented model risk framework, and sign-off from a risk/compliance function before being used for real credit decisions — this notebook is a prototype/demonstration, not a deployment-ready system.

## 25. Formulas Used — Quick Reference

A single place with every calculation used in this notebook, for quick reference when presenting.

**Filtering**
- Valid age: `18 <= age <= 90`
- Valid employment length: `employment_length <= (age - 14)`

**Dataset Create (harmonizing the two real datasets)**
- Annualize income: `annual_income = monthly_income x 12`
- Letter grade to credit score: `A=750, B=700, C=650, D=600, E=550, F=500, G=450`
- Age band to midpoint: e.g. `"25-34" -> 29.5`, `"<25" -> 22`, `">74" -> 80`
- Debt-to-income (Dataset 1 only): `debt_to_income_pct = loan_percent_income x 100`

**Clean & Preprocess**
- Median imputation: missing value replaced with that column's `median()`
- Outlier clipping: any value outside `[1st percentile, 99th percentile]` is clipped to that boundary

**Feature Engineering**
- `loan_to_income_ratio = loan_amount / annual_income`
- Min-max normalization: `credit_score_normalized = (credit_score - min) / (max - min) x 100`
- `high_dti_flag = 1 if debt_to_income_pct > 40 else 0`

**Feature Selection**
- Pearson correlation of each candidate feature with `target`
- Random Forest feature importance (Gini-based, averaged across 200 trees)

**Train / Test Split**
- 70% train / 30% test, stratified by `target`

**Models**
- Logistic Regression: `P(default) = 1 / (1 + e^-(b0 + b1*x1 + ... + bn*xn))`
- Decision Tree: recursive splits that minimize Gini impurity, max depth 6
- Random Forest: 300 trees, majority-vote / averaged probability, max depth 10

**Evaluation Metrics**
- `Accuracy = (TP + TN) / Total`
- `Precision = TP / (TP + FP)`
- `Recall = TP / (TP + FN)`
- `F1 = 2 x (Precision x Recall) / (Precision + Recall)`
- `ROC-AUC` = area under the True Positive Rate vs. False Positive Rate curve across all thresholds

**Cross-Validation**
- 5-fold stratified cross-validation, mean and standard deviation of ROC-AUC across folds

**Class Imbalance Handling**
- `class_weight="balanced"` reweights the minority class (defaults) during training

**Raw-Input Prediction (Section 18-20)**
- Same `loan_to_income_ratio`, `credit_score_normalized`, and `high_dti_flag` formulas above are applied live to whatever raw numbers are entered, before being passed into the trained models.

**Cost-Benefit Analysis (Section 21)**
- `cost_per_false_negative = average_loan_amount`
- `cost_per_false_positive = average_loan_amount x average_interest_rate`
- `Total Estimated Cost = (False Negatives x cost_per_false_negative) + (False Positives x cost_per_false_positive)`

**Risk Tiering (Section 22)**
- `Low risk: P(default) < 20%`
- `Medium risk: 20% <= P(default) < 50%`
- `High risk: P(default) >= 50%`

## 26. Summary

This notebook demonstrates a complete real-data loan-default classification pipeline:

**Filtering -> Dataset Create -> Realistic check -> Clean & Preprocess -> Feature Engineering -> Feature Selection -> Train/Test Split (70/30) -> 3 ML Models -> Evaluation -> Cross-Validation -> Class Imbalance Handling -> Feature Importance -> New-applicant prediction -> Cost-Benefit Analysis -> Risk Tiering -> Executive Summary -> Regulatory Note**

Key points:
- The dataset now combines **two real Kaggle datasets** (~170,000 rows) instead of synthetic data.
- Business framing was added on top of the ML pipeline: a cost-benefit analysis translates model errors into estimated money lost, risk tiering shows how predictions could drive real lending decisions, and an executive summary and compliance note frame the project for a non-technical/fintech audience.
- Every cleaning, imputation, and feature-engineering step printed its calculations (row counts, medians, clipping ranges, correlations, importances) so nothing is a hidden black box.
- `source` was deliberately excluded from modeling to prevent the model from simply learning which original dataset a row came from.
- The `results_df` table should be used to compare the models on the same held-out test set. No single metric should automatically determine a lending decision — the right metric depends on the business cost of false positives vs. false negatives.

**Responsible AI note:** While this now uses real datasets, the two sources were merged with several approximations (e.g. mapping letter grades to numeric scores, annualizing monthly income). A production lending system would need a single, consistent, well-governed data pipeline, fairness auditing across demographic groups, and regulatory compliance review before being used for real lending decisions.